# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment. Colab ships with DuckDB, so this is usually instant.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder.** Run locally, the notebook starts inside `work/notebooks/`, so it steps up until it reaches the top folder, where the `.env` file lives. On Colab it works in Colab's own folder.
2. **Loads the tools.** `duckdb` runs SQL directly on the warehouse files over the internet, `pandas` and `numpy` handle the small results, and `scipy` supplies the two probability models used for the luck test (`poisson` and `nbinom`, the negative binomial).
3. **Reads the Hugging Face token** from, in order: an environment variable, the project's `.env` file, a Colab Secret named `HF_TOKEN`, and only as a last resort a typed prompt. DuckDB then logs in with `PROVIDER credential_chain`, which reads the token from the environment itself, so the token never appears in a query, an output, or the notebook file.
4. **Defines the months.** Three development pairs are listed explicitly. June 2026 is never in any list, so no cell can read it.
5. **Defines `cached()`**, which runs a heavy warehouse query once, saves the small result as a Parquet file under `work/outputs/w03_cache/` (git ignores `*.parquet`), and reuses it on every later run. It also defines `show()` and `show_table()`, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys

import duckdb
import pandas as pd
from IPython.display import Markdown, display

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## 1. Unit of analysis + time window

**One row = one page in one month pair.** The page's search data in month M (used by the features and the score) is paired with its search data in month M+1 (used only by the target). The warehouse stores page-days; the notebook adds them up to page-months inside DuckDB, then pairs consecutive months.

| Pair   | Month M: features and score | Month M+1: target | Role                                   |
| ------ | --------------------------- | ----------------- | -------------------------------------- |
| 1      | February 2026               | March 2026        | Build the rules                        |
| 2      | March 2026                  | April 2026        | Check that the rules hold              |
| 3      | April 2026                  | May 2026          | Check that the rules hold              |
| Sealed | May 2026                    | June 2026         | Final test, never read while designing |

- **Why calendar months:** the daily table is stored as one folder per month, so each pair reads exactly two folders, and a month is the review cycle an editor works in.
- **Why these months:** the history is far from balanced. Most clients only arrive in the last eight months (the coverage table below shows it), so earlier months have too few clients to learn from.
- **Why June stays sealed:** June is the last month in the data, so it is the natural outcome month of any "what happens next" question. Designing rules while looking at it would mean designing inside the test.

**Output, in one sentence:** each month, the analysis hands an editor the 50 pages most likely to keep falling well short of the clicks their position should earn, with the clicks at stake and a reason code.

**Method:** this cell counts the rows of every warehouse table and compares them with the documented release. Row counts come from the Parquet file footers, so it reads no data values.


In [ ]:
# Documented sizes of build v20260703; a mismatch means the notebook is pointed at different data.
DOCUMENTED = {"dim_clients": 104, "dim_content": 519_606, "fact_content_daily_performance": 78_835_655,
              "fact_content_query_90d": 2_414_248}
SOURCES = {"dim_clients": DIM_CLIENTS, "dim_content": DIM_CONTENT, "fact_content_daily_performance": DAILY_FOOTERS,
           "fact_content_query_90d": QUERY_90D}
counts = {name: con.sql(f"SELECT COUNT(*) FROM {source}").fetchone()[0] for name, source in SOURCES.items()}
for name, rows in counts.items():
    assert rows == DOCUMENTED[name], f"{name}: {rows:,} rows, documented {DOCUMENTED[name]:,}"

show_table("Warehouse tables, counted against the documented release", pd.DataFrame({
    "Rows counted": [f"{counts[name]:,}" for name in DOCUMENTED],
    "Rows documented": [f"{rows:,}" for rows in DOCUMENTED.values()],
    "Match": ["yes"] * len(DOCUMENTED),
}, index=pd.Index(list(DOCUMENTED), name="Table")))

**Method:** this cell scans the daily table month by month, from January 2025 to May 2026 (June is left out on purpose), and counts how many clients and pages had at least one impression in each month, plus the first and last day with impressions. It runs once and is cached.


In [ ]:
# Clients and pages with impressions per month; the sums are kept to reconcile later tables against.
coverage = cached("coverage_by_month", f"""
    SELECT month,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages,
           SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
    FROM {month_files(PANEL_MONTHS)}
    WHERE gsc_impressions > 0
    GROUP BY month
    ORDER BY month""")
show_table("Clients and pages with search impressions, by month", pd.DataFrame({
    "First day": coverage["first_day"].dt.strftime("%Y-%m-%d").to_numpy(),
    "Last day": coverage["last_day"].dt.strftime("%Y-%m-%d").to_numpy(),
    "Clients": coverage["clients"].to_numpy(),
    "Pages": coverage["pages"].map("{:,}".format).to_numpy(),
}, index=pd.Index(coverage["month"], name="Month")))

## 2. Fields: feature / label / context / excluded

A data contract has to say what every field is for before any model sees it. Two levels matter:

- **Raw columns.** Every column of the three tables this project reads gets exactly one use, with a reason. The code below holds that list and refuses to run if any column is missing, listed twice, or new.
- **Row fields.** The fields built for each page-pair row go into exactly one of four buckets. **Feature:** known by the end of month M, safe to learn from. **Label:** the target or anything it is computed from. **Context:** for joining, grouping, splitting, and reading results, never learned from (IDs live here). **Excluded:** future information, product decisions, or fields that fail a check, each with a reason.

The same raw column can feed two buckets through different months. Clicks in month M are a feature; clicks in month M+1 are a label input. That is why row fields carry a `_m` or `_next` suffix, and why one check below rejects any feature ending in `_next`.

**Method:** this cell writes the raw-column contract as a table, then checks it against the actual column lists of the daily table (one month's folder is enough, since every month shares the same columns), the content table, and the client table.


In [ ]:
# Each entry: table, the columns it covers, how they are used, and why.
RAW_CONTRACT = [
    ("daily", ["report_date", "month"], "Window building", "Places each day in its month"),
    ("daily", ["client_hash_id", "content_hash_id"], "Context", "Pseudonymous IDs: joining, grouping, client-grouped splits"),
    ("daily", ["gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_avg_position"], "Feature source (M) and label source (M+1)",
     "Search Console measurements; month M builds features, month M+1 builds the target"),
    ("daily", ["gsc_data_available"], "Not used", "Only says whether the page had impressions that day (section 3c)"),
    ("daily", ["client_has_gsc"], "Not used", "Client eligibility uses dim_clients.gsc_data_start instead"),
    ("daily", ["client_has_ga4", "ga4_data_available"], "Context", "Mark when GA4 numbers are real; three-valued, filtered with IS TRUE"),
    ("daily", ["ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions", "ga4_total_engagement_sec",
               "scroll_events"], "Context", "Engagement reason code only; never ranks pages"),
    ("daily", ["sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid"], "Excluded",
     "Other traffic channels, outside a search click-through question"),
    ("daily", ["sessions_ai", "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other"],
     "Excluded", "AI-referral traffic: a different lane, and sparse"),
    ("content", ["client_hash_id", "content_hash_id", "keyword_hash_id", "url_hash_id"], "Context", "Pseudonymous IDs"),
    ("content", ["content_type", "main_intent", "search_volume", "competition", "cpc", "keyword_token_count"], "Feature",
     "Keyword research fields, fixed before the page is written; missing values get a has-value flag"),
    ("content", ["keyword_char_count", "competition_level"], "Excluded", "Duplicates of keyword_token_count and competition"),
    ("content", ["content_created_date", "keyword_created_date"], "Context", "Page and keyword age, for reading results"),
    ("content", ["content_updated_date", "word_count", "char_count", "backlinks", "category_count", "url_char_count"],
     "Excluded", "Snapshot values from July 2026 that may describe the page after later edits (section 3g)"),
    ("content", ["is_published", "is_deleted"], "Excluded", "The page's state in July 2026: future information"),
    ("content", ["provider_used", "model_used", "optimization_eligible_date"], "Excluded", "FlyRank product decisions"),
    ("content", ["last_optimized_date"], "Filter only", "Pages FlyRank optimized during a pair are removed from it"),
    ("clients", ["client_hash_id"], "Context", "Pseudonymous ID"),
    ("clients", ["gsc_data_start"], "Filter only", "A client's search history must cover all of month M"),
    ("clients", ["ga4_data_start", "has_gsc_access", "has_ga4_access", "access_profile"], "Context", "Coverage checks"),
    ("clients", ["is_active", "client_updated_date"], "Excluded", "Status at the July 2026 snapshot: future information"),
    ("clients", ["client_created_date"], "Not used", "Account creation says nothing about clicks"),
]
ACTUAL = {
    "daily": month_files(["2026-03"]),
    "content": DIM_CONTENT,
    "clients": DIM_CLIENTS,
}
for table, source in ACTUAL.items():
    listed = [column for entry in RAW_CONTRACT if entry[0] == table for column in entry[1]]
    actual = set(con.sql(f"DESCRIBE SELECT * FROM {source}").df()["column_name"])
    assert len(listed) == len(set(listed)), f"{table}: a column is listed twice."
    assert set(listed) == actual, f"{table}: unlisted {sorted(actual - set(listed))}, unknown {sorted(set(listed) - actual)}"

TABLE_NAME = {"daily": "fact_content_daily_performance", "content": "dim_content", "clients": "dim_clients"}
show_table("Raw-column contract (every column of the three tables, each listed once)", pd.DataFrame({
    "Table": [TABLE_NAME[entry[0]] for entry in RAW_CONTRACT],
    "Columns": [", ".join(entry[1]) for entry in RAW_CONTRACT],
    "Use": [entry[2] for entry in RAW_CONTRACT],
    "Why": [entry[3] for entry in RAW_CONTRACT],
}))

The fourth table, `fact_content_query_90d`, is **excluded as a source of features**, every column of it. It covers one fixed window, checked in section 3h. For the development pairs that window lies after month M and overlaps the target months, and no training pair has an equivalent. It stays available for one job: reading the top picks afterwards, for example to see whether a flagged page's impressions come from one dominant query.

**Method:** this cell writes the row-field buckets and checks the rules that keep them honest: no field in two buckets, no feature built from month M+1, and the planted leak from section 6 never among the features.


In [ ]:
# Row fields of one page-pair row; the suffix _m means month M and _next means month M+1.
FIVE_FEATURES = ["impressions_m", "clicks_m", "position_m", "position_volatility_m", "content_type"]
BUCKETS = {
    "Feature": FIVE_FEATURES + ["main_intent", "search_volume", "competition", "cpc", "keyword_token_count"],
    "Label": ["impressions_next", "clicks_next", "position_next", "expected_next", "p_next", "target"],
    "Context": ["client_hash_id", "content_hash_id", "month_m", "ga4_sessions_m"],
    "Excluded": ["position_two_months"],
}
everything = [field for fields in BUCKETS.values() for field in fields]
assert len(everything) == len(set(everything)), "A field sits in two buckets."
assert not [field for field in BUCKETS["Feature"] if field.endswith("_next")], "A feature is built from month M+1."
assert "position_two_months" not in BUCKETS["Feature"], "The planted leak is among the features."

WHY = {
    "Feature": "Known by the last day of month M",
    "Label": "Measured in month M+1; never an input",
    "Context": "Joining, grouping, splitting, reason codes; never learned from",
    "Excluded": "Position averaged over M and M+1: the overlap leak measured in section 6",
}
show_table("Row-field buckets (each field in exactly one)", pd.DataFrame({
    "Fields": [", ".join(fields) for fields in BUCKETS.values()],
    "Rule": [WHY[bucket] for bucket in BUCKETS],
}, index=pd.Index(list(BUCKETS), name="Bucket")))

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query here. The heavy part comes first: one scan per development month, adding page-days up to one row per page per month. The checks after it read that small table or a single month of daily rows.

### 3a. The page-month table

**Method:** this cell scans each of the four development months once and keeps, per page: impressions, clicks, the pieces needed for an impression-weighted position and its spread (summed only over days with a valid position, 1 or more, since Google positions start at 1), the clicks earned on days without a valid position, the number of days with impressions, and GA4 sessions counted only where `ga4_data_available IS TRUE`. Each month is cached as its own file. It then checks the grain and reconciles the totals against the coverage table.


In [ ]:
# One row per page per month; position pieces only from days with a valid position (1 or more).
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)

assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."
totals = page_months.groupby("month")[["impressions", "clicks"]].sum()
reference = coverage.set_index("month").loc[DEV_MONTHS, ["impressions", "clicks"]]
assert (totals == reference).all().all(), "Page-month totals do not reconcile with the coverage scan."

show_table("Page-month table: grain and reconciliation", pd.DataFrame({
    "Page rows": page_months.groupby("month").size().map("{:,}".format),
    "Impressions": totals["impressions"].map("{:,.0f}".format),
    "Matches coverage scan": "yes",
}).rename_axis("Month"))

### 3b. The daily grain

**Method:** this cell checks, on March 2026, that no page appears twice on the same date (the documented grain is date × client × page), and that no page belongs to more than one client. It also checks the content table's grain: one row per page.


In [ ]:
# Zero rows back from each probe means the grain holds.
march = month_files(["2026-03"])
daily_dupes = cached("grain_daily_march", f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n FROM {march}
    GROUP BY ALL HAVING COUNT(*) > 1 LIMIT 5""")
two_clients = cached("grain_page_client_march", f"""
    SELECT content_hash_id FROM {march}
    GROUP BY content_hash_id HAVING COUNT(DISTINCT client_hash_id) > 1 LIMIT 5""")
content_rows, content_pages = con.sql(f"SELECT COUNT(*), COUNT(DISTINCT content_hash_id) FROM {DIM_CONTENT}").fetchone()
assert daily_dupes.empty and two_clients.empty and content_rows == content_pages, "A grain check failed."

show("Grain checks (all passed)", {
    "pages appearing twice on one date (March)": len(daily_dupes),
    "pages belonging to two clients (March)": len(two_clients),
    "content table: rows vs distinct pages": f"{content_rows:,} vs {content_pages:,}",
})

### 3c. What the availability flags really mean

**Method:** this cell counts March's page-days by the two availability flags and, for each combination, how many have zero impressions or missing values. The documentation calls the flags three-valued (TRUE, FALSE, or empty).


In [ ]:
# Every flag combination with its row count and how its numbers look.
flags = cached("flag_states_march", f"""
    SELECT gsc_data_available, ga4_data_available, COUNT(*) AS days,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
           SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS empty_sessions
    FROM {march}
    GROUP BY ALL ORDER BY ALL""")
gsc_true = flags["gsc_data_available"].eq(True)
assert (flags.loc[gsc_true, "zero_impressions"] == 0).all(), "A TRUE search flag has a zero-impression day."
assert (flags.loc[~gsc_true, "zero_impressions"] == flags.loc[~gsc_true, "days"]).all(), "A FALSE search flag has impressions."

show_table("March 2026 page-days by availability flag", pd.DataFrame({
    "Search flag": flags["gsc_data_available"].map({True: "TRUE", False: "FALSE"}).to_numpy(),
    "GA4 flag": flags["ga4_data_available"].map({True: "TRUE", False: "FALSE"}).fillna("empty").to_numpy(),
    "Page-days": flags["days"].map("{:,}".format).to_numpy(),
    "With zero impressions": flags["zero_impressions"].map("{:,.0f}".format).to_numpy(),
    "With empty GA4 sessions": flags["empty_sessions"].map("{:,.0f}".format).to_numpy(),
}))

### 3d. Position traps

**Method:** this cell checks two things about position. First, whether `gsc_sum_position` is exactly the daily position × impressions (if so, summing it gives an exact impression-weighted position). Second, how many page-days have impressions but an impossible position: exactly 0, or between 0 and 1 (Google positions start at 1, so both mean the position was not recorded properly). Both kinds are left out of the position average, and the cell measures how much of the traffic sits on those days across the four development months.


In [ ]:
# Is sum_position exactly position x impressions, and how many days have impressions but an impossible position?
position_check = cached("position_checks_march_v2", f"""
    SELECT COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ABS(gsc_sum_position - gsc_avg_position * gsc_impressions) < 1e-6 THEN 1 ELSE 0 END) AS identity_holds,
           SUM(CASE WHEN gsc_avg_position <= 0 OR gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS position_zero_days,
           SUM(CASE WHEN gsc_avg_position > 0 AND gsc_avg_position < 1 THEN 1 ELSE 0 END) AS position_below_one_days,
           SUM(CASE WHEN gsc_avg_position > 0 AND gsc_avg_position < 1 THEN gsc_impressions ELSE 0 END) AS below_one_impressions,
           SUM(gsc_impressions) AS impressions
    FROM {march}
    WHERE gsc_impressions > 0""").iloc[0]
assert position_check["identity_holds"] == position_check["days_with_impressions"], "sum_position is not position x impressions."

share_impressions = 1 - page_months["position_impressions"].sum() / page_months["impressions"].sum()
share_clicks = page_months["clicks_without_position"].sum() / page_months["clicks"].sum()
show("Position checks", {
    "March days where sum_position = position x impressions": f"{position_check['identity_holds']:,.0f} of {position_check['days_with_impressions']:,.0f}",
    "March days with impressions but position 0": f"{position_check['position_zero_days']:,.0f}",
    "March days with a position between 0 and 1": f"{position_check['position_below_one_days']:,.0f}",
    "share of March impressions on days between 0 and 1": f"{position_check['below_one_impressions'] / position_check['impressions']:.2%}",
    "share of impressions on days without a valid position (Feb to May)": f"{share_impressions:.2%}",
    "share of clicks on days without a valid position (Feb to May)": f"{share_clicks:.2%}",
})

### 3e. Windows and client history

**Method:** this cell checks that each development month has data on its first and last calendar day, then counts, for each pair, the clients with impressions in month M and how many of them have search history (`gsc_data_start`) covering all of month M.


In [ ]:
# Each month's first and last day with impressions, from the coverage scan.
months = coverage.set_index("month").loc[DEV_MONTHS]
for month, row in months.iterrows():
    start = pd.Timestamp(f"{month}-01")
    assert row["first_day"] == start and row["last_day"] == start + pd.offsets.MonthEnd(0), f"{month} is not complete."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
rows = {}
for month_m, month_next in PAIRS:
    active = page_months.loc[page_months["month"] == month_m, "client_hash_id"].unique()
    covered = clients.loc[clients["client_hash_id"].isin(active) & (clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"))]
    rows[f"{month_m} → {month_next}"] = {"Clients with impressions in M": len(active), "History covers all of M": len(covered)}

show_table("Every development month is complete; clients whose history covers month M", pd.DataFrame(rows).T.rename_axis("Pair"))

### 3f. Missing values and their pattern

**Method:** this cell measures, for the keyword fields planned as features, how often each is empty, overall and separately for each content type. If the gaps follow content type, filling them with 0 would quietly teach a model the content type, so they get a has-value flag instead.


In [ ]:
# Share of empty values per keyword field, overall and by content type.
KEYWORD_FIELDS = ["search_volume", "competition", "cpc", "main_intent", "keyword_token_count"]
empty = cached("keyword_missing_by_type", f"""
    SELECT content_type, COUNT(*) AS pages,
           {", ".join(f"AVG(CASE WHEN {field} IS NULL THEN 1.0 ELSE 0 END) AS {field}" for field in KEYWORD_FIELDS)}
    FROM {DIM_CONTENT}
    GROUP BY content_type ORDER BY pages DESC""").set_index("content_type")
overall = empty[KEYWORD_FIELDS].mul(empty["pages"], axis=0).sum() / empty["pages"].sum()
table = pd.concat([overall.to_frame("All pages").T, empty[KEYWORD_FIELDS]])
show_table("Share of pages with an empty keyword field, by content type",
           table.map("{:.0%}".format).assign(Pages=[f"{empty['pages'].sum():,}"] + empty["pages"].map("{:,}".format).tolist())
           .rename_axis("Content type"))

**Method:** this cell measures how much of the development data has real GA4 engagement numbers, counting a page-month as covered only when it has GA4 sessions recorded on days where `ga4_data_available IS TRUE`.


In [ ]:
# GA4 coverage of page-months, which bounds how often an engagement reason code can apply.
ga4 = page_months.assign(has_ga4=page_months["ga4_sessions"] > 0, has_30=page_months["ga4_sessions"] >= 30)
show_table("Page-months with real GA4 sessions", pd.DataFrame({
    "Page-months": ga4.groupby("month").size().map("{:,}".format),
    "Any GA4 sessions": ga4.groupby("month")["has_ga4"].mean().map("{:.1%}".format),
    "30 or more sessions": ga4.groupby("month")["has_30"].mean().map("{:.1%}".format),
}).rename_axis("Month"))

### 3g. When the content table was captured

**Method:** this cell checks how many pages were edited after the first development month began, and in which months FlyRank's recorded optimizations fall. Both show whether content-table values can be trusted to describe a page as it was in month M.


In [ ]:
# How much of the content table changed after February 2026, and when optimizations happened.
snapshot = con.sql(f"""
    SELECT AVG(CASE WHEN content_updated_date >= DATE '2026-02-01' THEN 1.0 ELSE 0 END) AS edited_after,
           MAX(content_updated_date) AS last_edit,
           AVG(CASE WHEN last_optimized_date IS NOT NULL THEN 1.0 ELSE 0 END) AS has_optimization
    FROM {DIM_CONTENT}""").df().iloc[0]
optimized = con.sql(f"""
    SELECT strftime(last_optimized_date, '%Y-%m') AS month, COUNT(*) AS pages
    FROM {DIM_CONTENT} WHERE last_optimized_date IS NOT NULL GROUP BY ALL ORDER BY ALL""").df()

show("Content-table timing", {
    "pages edited on or after 2026-02-01": f"{snapshot['edited_after']:.1%}",
    "latest edit date in the table": f"{snapshot['last_edit']:%Y-%m-%d}",
    "pages with a recorded optimization": f"{snapshot['has_optimization']:.1%}",
})
show_table("Pages by month of their latest recorded optimization", optimized.assign(
    pages=optimized["pages"].map("{:,}".format)).set_index("month").rename_axis("Month").rename(columns={"pages": "Pages"}))

### 3h. The query table's window

**Method:** this cell reads the query table's window dates.


In [ ]:
# The query table covers one fixed window for every row.
window = con.sql(f"""
    SELECT MIN(window_start) AS first_start, MAX(window_start) AS last_start,
           MIN(window_end) AS first_end, MAX(window_end) AS last_end FROM {QUERY_90D}""").df().iloc[0]
assert window["first_start"] == window["last_start"] and window["first_end"] == window["last_end"], "The window is not fixed."
show("Query table window", {
    "window start": f"{window['first_start']:%Y-%m-%d}",
    "window end": f"{window['first_end']:%Y-%m-%d}",
    "same window on every row": "yes",
})

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.